In [ ]:
from IPython.display import HTML
HTML(open('../style.css', 'r').read())

In [ ]:
from typing import TypeVar
from collections.abc import Iterable

# Checking the Equivalence of Regular Expressions

## Type Checking

The functions in this notebook carry *type annotations*.  *Python* itself ignores these annotations, but the
type checker [*basedpyright*](https://docs.basedpyright.com) can use them to find errors before the program is
run.  In *JupyterLab*, the extension *jupyterlab-lsp* runs *basedpyright* in the background and underlines
type errors while you type.  On the command line, the command
```
basedpyright 09-Equivalence.ipynb
```
checks the whole notebook.  The settings of the type checker are stored in the file `pyrightconfig.json` in the
directory `Python`.

Both packages, `basedpyright` and `jupyterlab-lsp`, are installed by the script `fl.sh`.  Start `jupyter lab` in the directory `Python`, so that the settings in `pyrightconfig.json` are used.

In order to check whether two regular expressions $r_1$ and $r_2$ are *equivalent*, perform the 
following steps:
- convert the regular expressions $r_1$ and $r_2$ into *NFAs*
  $F_1$ and $F_2$ such that $L(r_1) = L(F_1)$ and $L(r_2) = L(F_2)$,
- convert the *NFAs* $F_1$ and $F_2$ into *DFAs*
  $D_1$ and $D_2$ such that $L(D_1) = L(F_1)$ and $L(D_2) = L(F_2)$
- check whether both $L(D_1) \backslash L(D_2)$ and $L(D_2) \backslash L(D_1)$ are empty.

The notebook `03-Regexp-2-NFA.ipynb` contains the method `RegExp2NFA.toNFA` that can be used to compute an
<span style="font-variant:small-caps;">Nfa</span> that accepts the language described by a given regular expression.

In [ ]:
%run 03-Regexp-2-NFA.ipynb

The notebook `01-NFA-2-DFA.ipynb` contains the function `nfa2dfa` that converts an <span style="font-variant:small-caps;">Nfa</span> into an equivalent <span style="font-variant:small-caps;">Dfa</span>.

In [ ]:
%run 01-NFA-2-DFA.ipynb

Given two sets `A` and `B`, the function `cartesian_product(A, B)` computes the 
<em style="color:blue">cartesian product</em> $A \times B$ which is defined as
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$

**Function `cartesian_product(A, B)`**
- *Input:* `A` and `B` are sets.
- *Output:* The Cartesian product $A \times B$.

In [ ]:
def cartesian_product[S, T](A: set[S], B: set[T]) -> set[tuple[S, T]]:
    return { (x, y) for x in A
                    for y in B
           }

In [ ]:
cartesian_product({1, 2}, {'a', 'b'})

In [ ]:
Char      = str
State     = TypeVar('State')
StatePair = tuple[State, State]
TransRel1 = dict[tuple[State, Char], State]
TransRel2 = dict[tuple[StatePair, Char], StatePair]
DFA1      = tuple[set[State], set[Char], TransRel1, State, set[State]]
DFA2      = tuple[set[StatePair], set[Char], TransRel2, StatePair, set[StatePair]]

Given two <span style="font-variant:small-caps;">Dfa</span>s `F1` and `F2`, the expression `fsm_complement(F1, F2)` computes a <span style="font-variant:small-caps;">Dfa</span>
that recognizes the language  $L(F_1)\backslash L(F_2)$.

**Function `fsm_complement(F1, F2)`**
- *Input:* `F1` and `F2` are complete <span style="font-variant:small-caps;">Dfa</span>s with the same alphabet.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> that accepts the language $L(F_1) \backslash L(F_2)$.

In [ ]:
def fsm_complement(F1: DFA1, F2: DFA1) -> DFA2:
    States1, Σ, 𝛿1, q1, A1 = F1
    States2, _, 𝛿2, q2, A2 = F2
    States = cartesian_product(States1, States2)
    𝛿 = {}
    for p1, p2 in States:
        for c in Σ:
            𝛿[(p1, p2), c] = (𝛿1[p1, c], 𝛿2[p2, c])
    return States, Σ, 𝛿, (q1, q2), cartesian_product(A1, States2 - A2)

In [ ]:
from typing import Literal

In [ ]:
type BinaryOp = Literal['⋅', '+']
type UnaryOp  = Literal['*']

In [ ]:
Char   = str
type RegExp = int | Char | tuple[RegExp, UnaryOp] | tuple[RegExp, BinaryOp, RegExp]

Given a regular expression $r$ and an alphabet $\Sigma$, the function $\texttt{regexp2DFA}(r, \Sigma)$
computes a *DFA* that accepts the language specified by $r$.

We have to use `# type: ignore`here because the class `RegExp2NFA`and the function `nfa2dfa` have been
defined in a different notebook.

**Function `regexp2DFA(r, Σ)`**
- *Input:* `r` is a regular expression and `Σ` is an alphabet.
- *Output:* A <span style="font-variant:small-caps;">Dfa</span> that accepts the language described by `r`.

In [ ]:
def regexp2DFA(r: RegExp, Σ: set[Char]) -> DFA1:
    converter = RegExp2NFA(Σ)       # type: ignore
    nfa       = converter.toNFA(r)
    dfa       = nfa2dfa(nfa)        # type: ignore
    return dfa # type: ignore

Given a <span style="font-variant:small-caps;">Dfa</span> $F$ the function 
`is_empty(F)` checks whether the language accepted by $F$ is empty.
In this function, the variable `Reachable` is the set of those states that are already known to be reachable
from the start state `q0`. `NewFound` are those states that can be reached from a state in the set 
`Reachable`.  When we find no new states that are reachable, the iteration stops and we check whether
there is a state that is both reachable and accepting because in that case the language is not empty.

**Function `is_empty(F)`**
- *Input:* `F` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* `True` if $L(F)$ is empty, `False` otherwise.

In [ ]:
def is_empty(F: DFA2) -> bool:
    States, Σ, δ, q0, Accepting = F
    Reachable = { q0 }
    while True:
        NewFound = { δ[q, c] for q in Reachable for c in Σ }
        if NewFound <= Reachable:
            break
        Reachable |= NewFound
    return Reachable & Accepting == set()

The function `regExpEquiv` takes three arguments:
- $r_1$ and $r_2$ are regular expressions,
- $\Sigma$ is the alphabet used in these regular expressions.

The function returns `True` iff $r_1 \doteq r_2$, i.e. if $r_1$ and $r_2$ are equivalent. 

**Function `regExpEquiv(r1, r2, Σ)`**
- *Input:* `r1` and `r2` are regular expressions and `Σ` is the alphabet used in them.
- *Output:* `True` if $r_1 \doteq r_2$, `False` otherwise.

In [ ]:
def regExpEquiv(r1: RegExp, r2: RegExp, Σ: set[Char]) -> bool:
    F1 = regexp2DFA(r1, Σ)
    F2 = regexp2DFA(r2, Σ)    
    r1_minus_r2 = fsm_complement(F1, F2)
    r2_minus_r1 = fsm_complement(F2, F1)
    return is_empty(r1_minus_r2) and is_empty(r2_minus_r1)

The notebook `10-Test-Equivalence.ipynb` can be used to test this function.